#HR-Tech AI: Интеллектуальный ИТ-глоссарий для отдела кадров и рекрутмента

Суть проекта: HR-специалисты и кадровики в Digital-компаниях постоянно сталкиваются со специфической ИТ-терминологией (DevOps, Agile, Kubernetes, фреймворки). Чтобы грамотно составлять профили должностей, грейдовые матрицы и онбординг-материалы для новичков, кадрам нужно быстро понимать суть технологий.
Наш Агент — это помощник для HR. Он получает сложный технический термин, лезет в википедию, вытаскивает суть и выдает понятную справку.

In [1]:
#установка и импорт библиотек (из-за конфликтов строго устанавливаем стабильные версии)
# Удаляем возможные конфликты
!!pip uninstall -y langchain langchain-community langchain-core
!pip install -q langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.43 wikipedia transformers accelerate torch

import time
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain.callbacks.base import BaseCallbackHandler

print("Библиотеки успешно загружены")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.
Библиотеки успешно загружены


In [2]:
# Класс для перехвата событий внутри агента
import time

class AgentMonitor:
    def __init__(self):
        self.start_time = 0
        self.end_time = 0
        self.llm_calls = 0
        self.tool_calls = 0

    def print_metrics(self):
        execution_time = self.end_time - self.start_time
        print("\n" + "="*50)
        print("ОТЧЕТ ТЕЛЕМЕТРИИ HR-АГЕНТА")
        print("="*50)
        print(f"Общее время выполнения: {execution_time:.2f} сек.")
        print(f"Циклов рассуждения (LLM): {self.llm_calls}")
        print(f"Обращений к внешним базам (Википедия): {self.tool_calls}")
        print("="*50 + "\n")

monitor = AgentMonitor()

In [3]:
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Загружаем LLM: {model_id} в формате float16.")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250,
    temperature=0.01,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000, lang="ru")
search_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print("Модель успешно загружена.")

Загружаем LLM: Qwen/Qwen2.5-1.5B-Instruct в формате float16.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_27649/3943917061.py:20: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  llm = HuggingFacePipeline(pipeline=pipe)


Модель успешно загружена.


In [4]:
import time

def run_custom_hr_agent(user_query):
    print(f"ЗАПРОС ОТ ОТДЕЛА КАДРОВ:\n{user_query}\n")

    monitor.start_time = time.time()
    monitor.llm_calls = 0
    monitor.tool_calls = 0

    print("Запуск аналитического процесса.")

    monitor.llm_calls += 1
    print("Шаг 1: Нейросеть выделяет термин для поиска.")

    extract_prompt = f"""<|im_start|>system
Извлеки из вопроса пользователя главное ИТ-понятие для поиска в Википедии. Напиши ТОЛЬКО этот термин на английском или русском, без лишних слов и знаков препинания.
Пример: "Найди инфу про DevOps" -> "DevOps"
<|im_end|>
<|im_start|>user
{user_query}<|im_end|>
<|im_start|>assistant
"""
    search_term = pipe(extract_prompt)[0]['generated_text'].strip()
    print(f"Термин для поиска: {search_term}")

    monitor.tool_calls += 1
    print("Шаг 2: Обращение к Википедии.")
    try:
        wiki_facts = search_tool.invoke(search_term)
        print("Факты получены.")
    except Exception as e:
        wiki_facts = "К сожалению, в Википедии нет точной статьи по этому запросу."
        print(f"Ошибка поиска: {e}")

    monitor.llm_calls += 1
    print("Шаг 3: Формирование HR-справки.")

    final_prompt = f"""<|im_start|>system
Ты — эксперт HR-Tech. Твоя задача — ответить на вопрос кадровика, опираясь СТРОГО на предоставленные факты из Википедии. Не придумывай ничего от себя.
Факты из Википедии:
{wiki_facts}
<|im_end|>
<|im_start|>user
{user_query}<|im_end|>
<|im_start|>assistant
"""
    final_answer = pipe(final_prompt)[0]['generated_text'].strip()

    monitor.end_time = time.time()
    monitor.print_metrics()

    return final_answer

test_query = "Для подготовки матрицы компетенций найди информацию про Agile. В каком году был опубликован Agile-манифест?"
final_answer = run_custom_hr_agent(test_query)

print(f"ФИНАЛЬНАЯ СПРАВКА ДЛЯ КАДРОВ:\n{final_answer}")

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ЗАПРОС ОТ ОТДЕЛА КАДРОВ:
Для подготовки матрицы компетенций найди информацию про Agile. В каком году был опубликован Agile-манифест?

Запуск аналитического процесса.
Шаг 1: Нейросеть выделяет термин для поиска.
Термин для поиска: Agile
Шаг 2: Обращение к Википедии.


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Факты получены.
Шаг 3: Формирование HR-справки.

ОТЧЕТ ТЕЛЕМЕТРИИ HR-АГЕНТА
Общее время выполнения: 255.57 сек.
Циклов рассуждения (LLM): 2
Обращений к внешним базам (Википедия): 1

ФИНАЛЬНАЯ СПРАВКА ДЛЯ КАДРОВ:
Agile методология была опубликована в 2001 году. Эта информация соответствует информации из Википедии о "Agile manifesto", который является основополагающим документом для Agile-методологии. Этот манифест содержит четыре ключевых идеи и десять принципов, которые стали основой Agile-методологии.
